# Notebook 02: Agent — Gene Lookup with Tools

**CABS AI Productivity Series**
Workshop: *LangChain, LangGraph & Local LLM Deployment: Building AI Agent Systems That Keep Your Data Safe*

---

## What is an Agent?

In Notebook 01, we built a RAG system: it retrieves text and generates answers. But it can only read documents you already loaded. It cannot go out and fetch new information.

An **Agent** is an LLM that can decide to **take actions** — like calling an API, querying a database, or running code. The key difference:

- **LLM**: you ask, it answers from what it knows
- **RAG**: you ask, it searches your documents, then answers
- **Agent**: you ask, it *decides what to do*, does it, reads the result, then answers

The agent loop:

User question → LLM thinks: "what do I need?" → Calls Tool A → Reads result → Maybe calls Tool B → Reads result → Writes final answer

### What you'll build in this notebook:

An agent with **two tools**:
1. **UniProt Gene Lookup** — fetches gene/protein info from the UniProt database
2. **AlphaFold Structure Lookup** — fetches predicted protein structure info from the AlphaFold database

The agent decides which tool(s) to call based on your question. Ask about gene function → it calls UniProt. Ask about protein structure → it calls AlphaFold. Ask about both → it chains both tools together.

### Why this matters for biologists:

- You already look up genes and structures manually every day — this automates that workflow
- The agent pattern generalizes: add tools for PubMed, BLAST, pathway databases, and your agent becomes a research assistant
- Understanding how agents call tools is essential for evaluating commercial "AI for biology" products

# Setup: Get Your Free Gemini API Key

1. Go to [aistudio.google.com](https://aistudio.google.com)
2. Sign in with your Google account
3. Accept Terms of Service
4. Left sidebar → **Get API Key** → **Create API key**
5. Copy the key (starts with `AIza...`)

No credit card needed. Run the cell below to enter your key.




In [ ]:
# ============================================================
# STEP 1: Install required packages
# ============================================================
# langchain              - framework for building LLM applications
# langchain-google-genai - connects LangChain to Google Gemini
# langgraph              - for building agent execution graphs
# requests               - for calling external APIs (UniProt, AlphaFold)

!pip install -q langchain langchain-google-genai langgraph requests

In [ ]:
# ============================================================
# STEP 2: Enter your Gemini API key
# ============================================================

import getpass
import os

api_key = getpass.getpass("Paste your Gemini API key here: ")
os.environ["GOOGLE_API_KEY"] = api_key

print("✅ API key set!")

In [ ]:
# ============================================================
# STEP 3: Quick test — make sure Gemini is working
# ============================================================

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

response = llm.invoke("What does the gene TP53 do? Answer in one sentence.")
print(response.content)
print("\n✅ Gemini is working!")

---
## Define the Tools

A **tool** is just a Python function that the agent can call. We wrap it with a decorator so LangChain knows:
- What the function does (from the docstring)
- What inputs it takes
- When to use it

The LLM reads these descriptions and decides which tool to call based on the user's question. This means **good docstrings matter** — they are instructions for the agent.

We will define two tools:
1. **UniProt lookup** — takes a gene name, returns protein info (function, location, disease associations)
2. **AlphaFold lookup** — takes a UniProt accession ID, returns predicted structure info and a link to view the 3D structure

In [ ]:
# ============================================================
# STEP 4: Define Tool 1 — UniProt Gene Lookup
# ============================================================
# UniProt is a public protein database. Its REST API is free,
# no API key required.
#
# This tool takes a gene name (like "TP53") and returns
# protein function, subcellular location, and disease associations.

import requests
from langchain_core.tools import tool

@tool
def lookup_gene_uniprot(gene_name: str) -> str:
    """Look up a gene/protein in the UniProt database.
    Use this tool when the user asks about gene function, protein
    information, disease associations, or subcellular location.
    Input should be a standard gene symbol like TP53, BRCA1, EGFR.
    Returns protein name, function, subcellular location, and
    disease involvement."""

    # Search UniProt for the gene name, filter to human (organism 9606)
    # and reviewed entries (Swiss-Prot) for high quality results
    search_url = "https://rest.uniprot.org/uniprotkb/search"
    params = {
        "query": f"(gene:{gene_name}) AND (organism_id:9606) AND (reviewed:true)",
        "format": "json",
        "size": 1,  # we only need the top result
        "fields": "accession,gene_names,protein_name,cc_function,cc_subcellular_location,cc_disease"
    }

    response = requests.get(search_url, params=params)

    if response.status_code != 200:
        return f"Error: UniProt API returned status {response.status_code}"

    data = response.json()
    results = data.get("results", [])

    if not results:
        return f"No results found for gene '{gene_name}' in UniProt (human, reviewed entries)."

    entry = results[0]

    # Extract key information
    accession = entry.get("primaryAccession", "N/A")

    # Protein name
    protein_name = "N/A"
    prot_desc = entry.get("proteinDescription", {})
    rec_name = prot_desc.get("recommendedName", {})
    if rec_name:
        protein_name = rec_name.get("fullName", {}).get("value", "N/A")

    # Gene names
    gene_names = [g.get("geneName", {}).get("value", "")
                  for g in entry.get("genes", [])]

    # Function
    function_texts = []
    for comment in entry.get("comments", []):
        if comment.get("commentType") == "FUNCTION":
            for text in comment.get("texts", []):
                function_texts.append(text.get("value", ""))

    # Subcellular location
    location_texts = []
    for comment in entry.get("comments", []):
        if comment.get("commentType") == "SUBCELLULAR LOCATION":
            for loc in comment.get("subcellularLocations", []):
                loc_val = loc.get("location", {}).get("value", "")
                if loc_val:
                    location_texts.append(loc_val)

    # Disease associations
    disease_texts = []
    for comment in entry.get("comments", []):
        if comment.get("commentType") == "DISEASE":
            disease = comment.get("disease", {})
            if disease:
                disease_texts.append(disease.get("diseaseId", ""))

    # Format the output
    output = f"""UniProt Entry: {accession}
Gene: {', '.join(gene_names) if gene_names else 'N/A'}
Protein: {protein_name}

Function: {' '.join(function_texts) if function_texts else 'No function annotation available.'}

Subcellular Location: {', '.join(location_texts) if location_texts else 'Not annotated.'}

Disease Associations: {', '.join(disease_texts) if disease_texts else 'None annotated.'}

UniProt URL: https://www.uniprot.org/uniprot/{accession}"""

    return output

print("✅ Tool defined: lookup_gene_uniprot")
print("   Looks up gene/protein info from UniProt")

In [ ]:
# ============================================================
# STEP 5: Test Tool 1 manually (before giving it to the agent)
# ============================================================
# Always test your tools independently first!
# This confirms the API works and the output format makes sense.

result = lookup_gene_uniprot.invoke({"gene_name": "TP53"})
print(result)

In [ ]:
# ============================================================
# STEP 6: Define Tool 2 — AlphaFold Structure Lookup
# ============================================================
# AlphaFold Protein Structure Database provides AI-predicted
# 3D structures for most known proteins. Free API, no key needed.
#
# This tool takes a UniProt accession ID and returns structure
# prediction info plus a link to view the 3D model.

@tool
def lookup_alphafold(uniprot_id: str) -> str:
    """Look up a protein's predicted 3D structure in the AlphaFold database.
    Use this tool when the user asks about protein structure, 3D structure,
    folding, or structural predictions.
    Input should be a UniProt accession ID like P04637 or Q9Y6K9.
    If you don't have the UniProt ID, use lookup_gene_uniprot first to find it.
    Returns structure prediction confidence and a link to view the 3D model."""

    url = f"https://alphafold.ebi.ac.uk/api/prediction/{uniprot_id}"

    response = requests.get(url)

    if response.status_code == 404:
        return f"No AlphaFold prediction found for UniProt ID '{uniprot_id}'."

    if response.status_code != 200:
        return f"Error: AlphaFold API returned status {response.status_code}"

    data = response.json()

    # AlphaFold returns a list; take the first entry
    if isinstance(data, list) and len(data) > 0:
        entry = data[0]
    else:
        return f"Unexpected response format from AlphaFold for '{uniprot_id}'."

    # Extract useful fields
    entry_id = entry.get("entryId", "N/A")
    gene = entry.get("gene", "N/A")
    organism = entry.get("organismScientificName", "N/A")
    confidence = entry.get("globalMetricValue", "N/A")
    model_url = entry.get("cifUrl", "N/A")
    pdb_url = entry.get("pdbUrl", "N/A")

    output = f"""AlphaFold Prediction: {entry_id}
Gene: {gene}
Organism: {organism}
Global Confidence (pLDDT): {confidence}

View 3D Structure: https://alphafold.ebi.ac.uk/entry/{uniprot_id}
Download PDB: {pdb_url}
Download mmCIF: {model_url}

Note: pLDDT > 90 = high confidence, 70-90 = good, 50-70 = low, < 50 = unreliable.
Different regions of the protein may have different confidence levels."""

    return output

print("✅ Tool defined: lookup_alphafold")
print("   Fetches predicted protein structure from AlphaFold DB")

In [ ]:
# ============================================================
# STEP 7: Test Tool 2 manually
# ============================================================
# P04637 is the UniProt accession for human TP53

result = lookup_alphafold.invoke({"uniprot_id": "P04637"})
print(result)

---
## Build the Agent

Now we give both tools to the LLM and let it decide when to use them.

This is the key conceptual leap:
- In RAG (Notebook 01), **you** decided the workflow: retrieve → read → answer
- With an agent, the **LLM** decides the workflow: "should I call UniProt? AlphaFold? Both? Neither?"

We use LangGraph's prebuilt ReAct agent, which implements this loop:

Receive question → Think about what to do → Call a tool (or answer directly) → Read the result → Think again → Maybe call another tool → Write final answer

"ReAct" stands for Reasoning + Acting — the agent alternates between thinking and doing.

In [ ]:
# ============================================================
# STEP 8: Create the agent with both tools
# ============================================================

from langgraph.prebuilt import create_react_agent

# Give the agent our two tools
tools = [lookup_gene_uniprot, lookup_alphafold]

# Create the agent
# The LLM will see the tool descriptions (the docstrings we wrote)
# and decide which tools to call based on the user's question
agent = create_react_agent(
    model=llm,
    tools=tools,
)

print("✅ Agent created with 2 tools:")
print("   1. lookup_gene_uniprot  — gene/protein info")
print("   2. lookup_alphafold     — predicted 3D structure")

---
## Talk to the Agent

Now ask questions in natural language. Watch how the agent decides which tools to call.

The key thing to observe: **you are not telling the agent which tool to use.** You just ask a question, and the agent figures out the right action.

In [ ]:
# ============================================================
# STEP 9: Helper function to run the agent and show its reasoning
# ============================================================
# This function prints each step the agent takes, so you can
# see the decision-making process — not just the final answer.

def ask_agent(question):
    print(f"❓ Question: {question}")
    print("=" * 60)

    # Stream the agent's steps so we can see what it does
    inputs = {"messages": [{"role": "user", "content": question}]}

    for step in agent.stream(inputs, stream_mode="values"):
        messages = step["messages"]
        last_msg = messages[-1]

        # Show tool calls
        if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
            for tc in last_msg.tool_calls:
                print(f"\n🔧 Calling tool: {tc['name']}")
                print(f"   Input: {tc['args']}")

        # Show tool results
        elif last_msg.type == "tool":
            content = last_msg.content
            # Truncate long tool outputs for readability
            if len(content) > 500:
                content = content[:500] + "..."
            print(f"\n📋 Tool result (truncated):\n{content}")

        # Show final answer
        elif last_msg.type == "ai" and not hasattr(last_msg, "tool_calls"):
            pass  # will print below
        elif last_msg.type == "ai" and not last_msg.tool_calls:
            pass  # will print below

    # Print the final response
    final = step["messages"][-1].content
    print(f"\n{'=' * 60}")
    print(f"💬 Final Answer:\n{final}")

In [ ]:
# ============================================================
# STEP 10: Ask about gene function (should call UniProt only)
# ============================================================

ask_agent("What does the BRCA1 gene do and what diseases is it associated with?")

In [ ]:
# ============================================================
# STEP 11: Ask about protein structure (should call both tools)
# ============================================================
# Watch: the agent should first call UniProt to get the accession ID,
# then call AlphaFold with that ID. This is tool chaining.

ask_agent("What is the predicted 3D structure confidence for the EGFR protein?")

In [ ]:
# ============================================================
# STEP 12: Ask a combined question (should call both tools)
# ============================================================

ask_agent("Tell me about the TP53 protein — its function, disease associations, and how confident is the AlphaFold structure prediction?")

In [ ]:
# ============================================================
# STEP 13: Ask something the tools can't answer
# ============================================================
# A well-behaved agent should recognize its tools don't cover this
# and answer from its own knowledge or say it doesn't know.

ask_agent("What is the best cell line to use for a lung-on-a-chip experiment?")

In [ ]:
# ============================================================
# STEP 14: Try your own question!
# ============================================================
# Some ideas:
#   "What is the function of the ACE2 receptor?"
#   "Look up the insulin gene and its AlphaFold structure"
#   "What diseases are associated with the CFTR gene?"
#   "Tell me about KRAS — function, location, and structure confidence"

my_question = "Look up the insulin gene and its AlphaFold structure"

ask_agent(my_question)

---
## What Just Happened?

The agent did something fundamentally different from the RAG system in Notebook 01:

- **RAG** reads documents you already have. The knowledge is static — it's whatever you loaded.
- **The agent** goes out and fetches live data from external databases. The knowledge is dynamic.

And critically, the agent **decided on its own** which tools to call and in what order. When you asked about structure, it figured out it needed the UniProt ID first, called UniProt, extracted the accession, then called AlphaFold. You didn't program that sequence — the LLM reasoned through it.

### This is the agent pattern:

LLM (brain) + Tools (hands) + Decision loop (autonomy) = Agent

### What you could add:

The same pattern extends to any API or database:
- A PubMed search tool → agent retrieves latest papers
- A BLAST tool → agent runs sequence alignment
- A pathway database tool (KEGG, Reactome) → agent finds pathway membership
- A lab inventory tool → agent checks reagent availability

Each tool is just a Python function with a good description. The agent learns when to use it from the docstring.

### But — where did your data go?

Every question you asked and every tool result went through Google's Gemini API. The gene names aren't sensitive, but the pattern matters: if you were asking about unpublished gene targets or proprietary sequences, that data would leave your machine.

**Notebook 03** shows how to run the same agent with a local LLM using Ollama — so your queries and reasoning stay on your computer.

👉 Continue to [03_local_llm_deployment_with_ollama.md](https://github.com/lecaibio/cabs-workshop-llm-agents/blob/main/notebooks/03_local_llm_deployment_with_ollama.md)